In [ ]:
!pip install pyngrok

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from peft import PeftModel

# ========================
# CONFIG
# ========================
NGROK_AUTH_TOKEN = "31SqEHXrrnGrHl3csmIxh3hLYHD_4Cofe3qsgjiiPjuAtMTNX"
USE_ADAPTER = True
#BASE_MODEL_PATH = "AITeamVN/GRPO-VI-Qwen2-7B-RAG"
BASE_MODEL_PATH = "Qwen/Qwen3-4B-Instruct-2507"
ADAPTER_PATH = "/kaggle/input/tune-v4/ft4b_8m"

# "AITeamVN/GRPO-VI-Qwen2-7B-RAG" 
# "Qwen/Qwen3-4B-Instruct-2507"
# "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
# ========================

# Ngrok setup
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000)
print("Public URL:", public_url)

# Flask app
app = Flask(__name__)


# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH,
    trust_remote_code=True
)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype="auto",
    device_map="auto"
)
base_model.resize_token_embeddings(len(tokenizer))

# Load adapter nếu có
if USE_ADAPTER:
    print("🔹 Loading model with adapter...")
    model = PeftModel.from_pretrained(
        base_model,
        ADAPTER_PATH,
    )
else:
    print("🔹 Loading model WITHOUT adapter...")
    model = base_model


# ========================
# PREDEFINED INSTRUCTIONS (STRICT MATH STYLE)
# ========================
MATH_STRICT_INSTRUCTION = """
    Bạn là một trợ lý toán học, chuyên giải quyết các bài toán và câu hỏi của người dùng. 
    Khi trình bày bạn phải
      + mỗi công thức, Tiêu đề,... phải ở trên một dòng riêng.
      + Không dùng inline LaTeX.
      + Ngắn gọn, chỉ giữ bước quan trọng, không văn hoa.
"""

STYLES = {
    # Phong cách mặc định cho các bài toán (đúng yêu cầu người dùng)
    "bai_toan": MATH_STRICT_INSTRUCTION
}


# API endpoint
@app.route('/generate', methods=['POST'])
def generate():
    data = request.get_json()
    prompt = data.get("prompt", "")
    if not prompt:
        return jsonify({"error": "Empty prompt"}), 400
    instruction = STYLES["bai_toan"]
    
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user",   "content": prompt}
    ]

    # Áp dụng chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.3,
        top_p=0.95,
        top_k=50,
        do_sample=True
    )

    
    output_ids = generated[0][len(inputs.input_ids[0]):].tolist()
    content = tokenizer.decode(output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

    return jsonify({"response": content})

Public URL: NgrokTunnel: "https://c0da5d7311ce.ngrok-free.app" -> "http://localhost:5000"


config.json:   0%|          | 0.00/709 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

🔹 Loading model with adapter...


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

In [ ]:
if __name__ == "__main__":
    app.run(port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


# TEST MODEL

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen3-1.7B-Base"
# "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# Load tokenizer và model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
prompt = """
    Dựa trên thông tin sau, hãy trả lời câu hỏi, 
    Khi trả lời, hãy tuân thủ nghiêm ngặt các quy định sau:
    - Nếu có sử dụng công thức toán học, hãy đặt công thức trong khối LaTeX dùng `$$` (hoặc trong code block Markdown nếu cần).
    - Không cần nói "Dựa trên thông tin được cung cấp"
    - Mỗi bước xuống dòng riêng.\n"    
    Thông tin:
     Tập con khác rỗng F của không gian vector V là một không gian con của V khi và chỉ khi hai điều kiện sau thỏa:  \ni) $\\forall x, y \\in F : x + y \\in F$.  \nii) $\\forall x \\in F, \\alpha \\in K : \\alpha x \\in F$.  \n**Ví dụ 4.23** Cho $F = \\{(x_1; x_2; x_3) \\in R^3 | x_1 + 2x_2 - x_3 = 0 \\}$.  \na) Chứng tỏ F là không gian con của $R^3$.  \nb) Tìm cơ sở và số chiều của F.  \n**Bài làm**  \na) Sinh viên tự kiểm tra 2 điều kiện trong định lý.  \nb) $\\forall x = (x_1; x_2; x_3) \\in F \\Longleftrightarrow x_1 + 2x_2 - x_3 = 0 \\Longleftrightarrow x_3 = x_1 + 2x_2$.  \n$x = (x_1; x_2; x_3) = (x_1; x_2; x_1 + 2x_2) = x_1(1; 0; 1) + x_2(0; 1; 2)$.  \nSuy ra $E = \\{(1,0,1), (0,1,2)\\}$ là tập sinh của F.  \nKiểm tra E độc lập tuyến tính. Vậy E là cơ sở của F và $\\dim(F) = 2$.  \n**Ví dụ 4.24** Cho $F = \\{p(x) \\in P_2[x] | p(1) = 0 \\land p(2) = 0\\}$.  \na) Chứng tỏ F là không gian con của $P_2[x]$.  \nb) Tìm cơ sở và số chiều của F.  \n**Bài làm**  \na) Sinh viên tự kiểm tra 2 điều kiện trong định lý.  \nb) $\\forall p(x) = ax^2 + bx + c \\in F \\Longleftrightarrow p(1) = 0 \\land p(2) = 0$  \n$ \\iff \\begin{cases} a+b+c=0 \\\\ 4a+2b+c=0 \\end{cases} \\iff \\begin{cases} a=\\alpha \\\\ b=-3\\alpha \\\\ c=2\\alpha \\end{cases} $  \n$p(x) = \\alpha x^2 - 3\\alpha x + 2\\alpha = \\alpha(x^2 - 3x + 2)$.  \nSuy ra $E = \\{x^2 - 3x + 2\\}$ là tập sinh của F.  \nHiển nhiên E độc lập tuyến tính. Vậy E là cơ sở của F và $\\dim(F) = 1$.  \n**Ví dụ 4.25** Cho $F = \\left\\{ A \\in M_2[R] | A \\begin{pmatrix} 1 & -1 \\\\ 2 & -2 \\end{pmatrix} = \\begin{pmatrix} 0 & 0 \\\\ 0 & 0 \\end{pmatrix} \\right\\}.$  \na) Chứng tỏ F là không gian con của $M_2[R]$.  \nb) Tìm cơ sở và số chiều của F.  \n**Bài làm**  \na) Sinh viên tự kiểm tra 2 điều kiện trong định lý.  \nb) $\\forall A = \\begin{pmatrix} a & b \\\\ c & d \\end{pmatrix} \\in F \\Longleftrightarrow \\begin{pmatrix} a & b \\\\ c & d \\end{pmatrix} \\begin{pmatrix} 1 & -1 \\\\ 2 & -2 \\end{pmatrix} = \\begin{pmatrix} 0 & 0 \\\\ 0 & 0 \\end{pmatrix} \\Longleftrightarrow \\begin{pmatrix} a+2b & -a-2b \\\\ c+2d & -c-2d \\end{pmatrix} = \\begin{pmatrix} 0 & 0 \\\\ 0 & 0 \\end{pmatrix}$  \n$ \\iff \\begin{cases} a+2b=0 \\\\ c+2d=0 \\end{cases} \\iff \\begin{cases} a=-2b \\\\ c=-2d \\end{cases} $  \n$ A = \\begin{pmatrix} -2b & b \\\\ -2d & d \\end{pmatrix} = b \\begin{pmatrix} -2 & 1 \\\\ 0 & 0 \\end{pmatrix} + d \\begin{pmatrix} 0 & 0 \\\\ -2 & 1 \\end{pmatrix}. $  \nSuy ra $E = \\left\\{ \\begin{pmatrix} -2 & 1 \\\\ 0 & 0 \\end{pmatrix}, \\begin{pmatrix} 0 & 0 \\\\ -2 & 1 \\end{pmatrix} \\right\\}$ là tập sinh của F.  \nDễ thấy E độc lập tuyến tính. Vậy E là cơ sở của F và $\\dim(F) = 2$.\n Trong không gian vector V, nếu tập con F với các phép toán trong V lập thành một không gian vector thì ta nói F là không gian con của V.\n Trong $R^n$, cho không gian con F:  \n**TH 1)** Cho F được cho bởi tập sinh $F = \\langle v_1, v_2, \\ldots, v_m \\rangle:$  \nLập ma trận có các vector $v_i$ là hàng $A = \\begin{pmatrix} v_1 \\\\ v_2 \\\\ \\vdots \\\\ v_m \\end{pmatrix} \\xrightarrow{bdsc} \\text{ ma trận bậc thang}$  \n$\\dim(F) = r(A)$ và cơ sở gồm các hàng khác không của ma trận bậc thang.  \n**TH 2)** Cho F là tập nghiệm của hệ phương trình tuyến tính thuần nhất $AX = 0$:  \nGiải hệ. Số chiều của không gian nghiệm là $\\dim(F) = n - r(A)$ (với n là số ẩn) và cơ sở được suy ra từ nghiệm tổng quát của hệ.  \n**Ví dụ 4.27** Trong $R^3$, cho tập $M = \\{(1, 1, 1), (2, 3, 1), (1, 0, 2)\\}$.  \na) $x = (1, -2, 3)$ thuộc không gian con span(M) hay không?  \nb) Tìm m để $x = (1, 0, m) \\in \\text{span}(M)$.  \n**Bài làm**  \na) x thuộc không gian con span(M) khi và chỉ khi x là tổ hợp tuyến tính của các vector trong M. Ta lập ma trận cột:  \n$ [M|x] = \\left[ \\begin{array}{ccc|c} 1 & 2 & 1 & 1 \\\\ 1 & 3 & 0 & -2 \\\\ 1 & 1 & 2 & 3 \\end{array} \\right] \\xrightarrow{bdsc} \\left[ \\begin{array}{ccc|c} 1 & 2 & 1 & 1 \\\\ 0 & 1 & -1 & -3 \\\\ 0 & 0 & 0 & -1 \\end{array} \\right] $  \n$r(M) = 2 < r([M|x]) = 3 \\Longrightarrow x \\notin \\text{span}(M)$.  \nb) $ [M|x] = \\left[ \\begin{array}{ccc|c} 1 & 2 & 1 & 1 \\\\ 1 & 3 & 0 & 0 \\\\ 1 & 1 & 2 & m \\end{array} \\right] \\xrightarrow{bdsc} \\left[ \\begin{array}{ccc|c} 1 & 2 & 1 & 1 \\\\ 0 & 1 & -1 & -1 \\\\ 0 & 0 & 0 & m-2 \\end{array} \\right] $  \n$x \\in \\text{span}(M) \\Longleftrightarrow r(M) = r([M|x]) \\Longleftrightarrow m-2 = 0 \\Longleftrightarrow m = 2$.\n Cho $M = \\{v_1, v_2, \\ldots, v_p\\} \\subset V$.  \nKý hiệu $H := \\text{Span}(M) = \\{\\alpha_1v_1 + \\alpha_2v_2 + \\cdots + \\alpha_pv_p | \\forall \\alpha_i \\in R\\}$.  \n• H là một không gian con được sinh bởi M: $H = \\langle M \\rangle$.  \n• $\\dim(H) = \\text{r}(M)$.  \n• $x \\in H \\Longleftrightarrow x$ là tổ hợp tuyến tính của $M \\Longleftrightarrow \\text{r}(M, x) = \\text{r}(M)$.  \n**Ví dụ 4.26** Tìm cơ sở và số chiều của các không gian con sau  \na) $F = \\langle (1;1;1), (2;1;1), (3;1;1) \\rangle$.  \nb) $F = \\langle x^2 + x + 1, 2x^2 + 3x - 1, x^2 + 2x - 2 \\rangle$.  \nc) $F = \\left\\langle \\begin{pmatrix} 1 & 1 \\\\ 2 & 1 \\end{pmatrix}, \\begin{pmatrix} 2 & 1 \\\\ 0 & 1 \\end{pmatrix}, \\begin{pmatrix} 3 & 1 \\\\ -2 & 1 \\end{pmatrix}, \\begin{pmatrix} 1 & 0 \\\\ -2 & 0 \\end{pmatrix} \\right\\rangle$  \nd) $F = \\{(x_1; x_2; x_3; x_4) \\in R^4 | x_1 + x_2 + x_3 = 0 \\land x_1 - x_2 + x_4 = 0 \\}$  \n**Bài làm**  \na) $A = \\begin{pmatrix} 1 & 1 & 1 \\\\ 2 & 1 & 1 \\\\ 3 & 1 & 1 \\end{pmatrix} \\xrightarrow{bdsc} \\begin{pmatrix} 1 & 1 & 1 \\\\ 0 & -1 & -1 \\\\ 0 & 0 & 0 \\end{pmatrix} \\Longrightarrow \\dim(F) = r(A) = 2$ và cơ sở của F là $\\{(1; 1; 1), (0; -1; -1)\\}$.  \nb) $A = \\begin{pmatrix} 1 & 1 & 1 \\\\ 2 & 3 & -1 \\\\ 1 & 2 & -2 \\end{pmatrix} \\xrightarrow{bdsc} \\begin{pmatrix} 1 & 1 & 1 \\\\ 0 & 1 & -3 \\\\ 0 & 0 & 0 \\end{pmatrix} \\implies \\dim(F) = r(A) = 2$ và cơ sở của F là $\\{x^2 + x + 1, x - 3\\}$.  \nc) $A = \\begin{pmatrix} 1 & 1 & 2 & 1 \\\\ 2 & 1 & 0 & 1 \\\\ 3 & 1 & -2 & 1 \\\\ 1 & 0 & -2 & 0 \\end{pmatrix} \\xrightarrow{bdsc} \\begin{pmatrix} 1 & 1 & 2 & 1 \\\\ 0 & -1 & -4 & -1 \\\\ 0 & 0 & 0 & 0 \\\\ 0 & 0 & 0 & 0 \\end{pmatrix} \\implies \\dim(F) = 2 \\text{ và cơ sở của } F \\text{ là } \\left\\{ \\begin{pmatrix} 1 & 1 \\\\ 2 & 1 \\end{pmatrix}, \\begin{pmatrix} 0 & -1 \\\\ -4 & -1 \\end{pmatrix} \\right\\}.$  \nd) Giải hệ $\\begin{cases} x_1 + x_2 + x_3 = 0 \\\\ x_1 - x_2 + x_4 = 0 \\end{cases} \\Longleftrightarrow \\begin{bmatrix} 1 & 1 & 1 & 0 \\\\ 1 & -1 & 0 & 1 \\end{bmatrix} \\begin{bmatrix} x_1 \\\\ x_2 \\\\ x_3 \\\\ x_4 \\end{bmatrix} = \\begin{bmatrix} 0 \\\\ 0 \\end{bmatrix} \\xrightarrow{bdsc} \\begin{bmatrix} 1 & 1 & 1 & 0 \\\\ 0 & -2 & -1 & 1 \\end{bmatrix} \\begin{bmatrix} x_1 \\\\ x_2 \\\\ x_3 \\\\ x_4 \\end{bmatrix} = \\begin{bmatrix} 0 \\\\ 0 \\end{bmatrix}$  \nĐặt $x_3 = 2\\alpha$, $x_4 = 2\\beta$.  \nTừ pt(2): $-2x_2 -x_3 + x_4 = 0 \\implies x_2 = \\frac{1}{2}(-x_3 + x_4) = -\\alpha + \\beta$.  \nTừ pt(1): $x_1 = -x_2 - x_3 = -(-\\alpha + \\beta) - 2\\alpha = -\\alpha - \\beta$.  \n$\\forall x \\in F \\Longleftrightarrow x = (-\\alpha - \\beta; -\\alpha + \\beta; 2\\alpha; 2\\beta) = \\alpha(-1; -1; 2; 0) + \\beta(-1; 1; 0; 2)$.  \nSuy ra $E = \\{(-1, -1, 2, 0), (-1, 1, 0, 2)\\}$ là tập sinh của F.  \nDễ thấy E độc lập tuyến tính. Vậy E là cơ sở của F và $\\dim(F) = 2$.\n $F \\cap G \\subset F, G \\subset F + G \\subset V$."
    Câu hỏi:
    Cho không gian con $W$ của $\\mathbb{R}^3$ được xác định bởi phương trình $x + y - z = 0$. Tìm cơ sở và số chiều của $W$.
    Trả lời:
"""
messages = [{"role": "user", "content": prompt}]

# Áp dụng chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Sinh text
generated = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=True
)
output_ids = generated[0][len(inputs.input_ids[0]):].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

print("Output:", content)